In [183]:
import os
import numpy as np
import pandas as pd
import datasets
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple
from datetime import datetime

def load_dataset(data_path: str) -> pd.DataFrame:
    """
    Load dataset from Parquet files and convert to a Pandas DataFrame.
    """
    dataset = datasets.Dataset.from_parquet(os.path.join(data_path, 'data/*'))
    return dataset.to_pandas()

def sample_subjects(df: pd.DataFrame, fraction: float = 0.1, seed: int = 42) -> pd.DataFrame:
    """
    Randomly sample a fraction of unique subjects from the dataset.
    """
    unique_subjects = df['subject_id'].unique()
    sampled_subjects = np.random.choice(unique_subjects, size=int(len(unique_subjects) * fraction), replace=False)
    return df[df['subject_id'].isin(sampled_subjects)]

def compute_event_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the number of events per subject.
    """
    event_counts = df['subject_id'].value_counts().reset_index()
    return event_counts.rename(columns={'index': 'subject_id', 'subject_id': 'event_count'})

def plot_event_distribution(event_counts: pd.DataFrame, bins: int = 30) -> None:
    """
    Plot the distribution of event counts per subject.
    """
    plt.figure(figsize=(10, 5))
    plt.hist(event_counts['event_count'], bins=bins, edgecolor='black')
    plt.xlabel("Number of Events per Subject")
    plt.ylabel("Frequency")
    plt.title("Distribution of Events per Subject")
    plt.grid(axis='y', alpha=0.75)
    plt.show()

def filter_demographics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract demographic data (age, sex, race, ethnicity) for each subject.
    Excludes subjects with missing demographics and filters out those under 18 or over 89.
    """
    # Filter for demographic data
    demo_df = df[df['table'] == 'person'][['subject_id', 'code', 'time']]
    demo_df['text_value'] = demo_df['code'].str.split('/').str[1]
    demo_df['code'] = demo_df['code'].str.split('/').str[0]
    demo_df = demo_df[demo_df['code'].isin(['Ethnicity', 'Gender', 'Race'])]
    
    age_df = df[df['code'] == 'MEDS_BIRTH'][['subject_id', 'code', 'time']]
    age_df['code'] = 'Age'
    age_df['text_value'] = (pd.Timestamp.today() - pd.to_datetime(age_df['time'])).dt.days // 365
    
    # Merge demographic data with age data
    demo_df = pd.concat([demo_df, age_df])
    demo_df = demo_df.pivot(index='subject_id', columns='code', values='text_value').reset_index()
    demo_df = demo_df.dropna(subset=['Gender', 'Age', 'Race', 'Ethnicity'])
    demo_df = demo_df[(demo_df['Age'] >= 18) & (demo_df['Age'] <= 89)]
    
    return demo_df

def standardize_and_compute_bmi(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardizes the weight and height data, and computes BMI where missing for each time point.
    Assumes some values might be in pounds or ounces and converts accordingly.
    If BMI is missing, it will be calculated using weight and height.
    """
    
    def infer_and_convert_weight(row):
        if pd.isna(row['unit']):
            if row['numeric_value'] > 1000: 
                row['unit'] = 'ounces'
            elif row['numeric_value'] > 100:  
                row['unit'] = 'lbs'
            else:
                row['unit'] = 'kg'
        
        if row['unit'] == 'ounces':
            row['numeric_value'] *= 0.0283495
        elif row['unit'] == 'lbs':
            row['numeric_value'] *= 0.453592

        row['unit'] = 'kg'
        return row
    
    weight_df = df[df['code'] == "LOINC/29463-7"].apply(infer_and_convert_weight, axis=1)
    height_df = df[df['code'] == "LOINC/8302-2"]
    height_df['numeric_value'] *= 0.0254
    height_df['unit'] = 'meters'
    
    merged_df = pd.merge(weight_df[['subject_id', 'time', 'numeric_value', 'visit_id']], 
                         height_df[['subject_id', 'time', 'numeric_value', 'visit_id']], 
                         on=['subject_id', 'time', 'visit_id'], how='left', suffixes=('_weight', '_height'))

    merged_df['BMI_computed'] = merged_df.apply(
        lambda row: row['numeric_value_weight'] / (row['numeric_value_height'] ** 2) 
        if pd.notna(row['numeric_value_weight']) and pd.notna(row['numeric_value_height']) 
        else None, axis=1)
    merged_df = merged_df.dropna(subset=['BMI_computed'])
    
    print('Number of events with existing BMI:', df[df['code'] == "LOINC/39156-5"][['subject_id', 'time']].drop_duplicates().shape[0])
    print('Number of events with computed BMI but no existing BMI:',
        len(set(merged_df[['subject_id', 'time']].drop_duplicates().apply(tuple, axis=1))
            .difference(set(df[df['code'] == "LOINC/39156-5"][['subject_id', 'time']].drop_duplicates().apply(tuple, axis=1)))))
        
    bmi_df = df[df['code'] == "LOINC/39156-5"].rename(columns={'numeric_value': 'BMI_existing'})
    merged_bmi = pd.merge(merged_df[['subject_id', 'time', 'visit_id', 'BMI_computed']], 
                          bmi_df[['subject_id', 'time', 'visit_id', 'BMI_existing']], 
                          on=['subject_id', 'time', 'visit_id'], how='outer')
    merged_bmi['final_bmi'] = merged_bmi['BMI_existing'].fillna(merged_bmi['BMI_computed'])
    return merged_bmi[['subject_id', 'time', 'visit_id', 'final_bmi']]

unit_conversion_dict = {
    'g/dL': 'g/dL',
    'G/DL': 'g/dL',
    'g/dl': 'g/dL',
    '%': '%',
    'None': 'None',
    'K/uL': 'K/uL',
    'KUL': 'K/uL',
    'x10E3/uL': 'K/uL',  
    '10x3/uL': 'K/uL',   
    'pg': 'pg',
    'PG': 'pg',
    'fL': 'fL',
    'FL': 'fL',
    'fl': 'fL',
    'Thousand/uL': 'K/uL',
    'Million/uL': 'Million/uL',  
    'MUL': 'Million/uL',
    'MIL/uL':'Million/uL',
    '10*6/uL': 'Million/uL',
    '10x6/uL': 'Million/uL',
    'M/uL': 'Million/uL',
    'x10E6/uL': 'Million/uL',
}

CBC_LOINC_CODES = [
    "LOINC/718-7", "LOINC/4544-3", "LOINC/789-8", "LOINC/777-3", "LOINC/785-6", "LOINC/786-4", 
    "LOINC/787-2", "LOINC/788-0"
]

def extract_cbc_tests(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters the dataset to only include CBC lab results based on LOINC codes.
    """
    df = df[df['table'] == 'measurement']
    return df[df['code'].isin(CBC_LOINC_CODES)]

def check_units(cbc_df: pd.DataFrame, data_source: str = "EHRSHOT") -> None:
    """
    Check if there are multiple units per lab test and print out the number of units.
    """
    print("Checking for multiple units per lab test...")
    cbc_df['unit'] = cbc_df['unit'].replace(unit_conversion_dict)
    lab_units = cbc_df[['code', 'unit']].drop_duplicates()

    if data_source == "EHRSHOT":
        for lab_code in lab_units['code'].unique():
            lab_data = lab_units[lab_units['code'] == lab_code]
            num_units = len(lab_data)
            if num_units > 1:
                print(f"Warning: Lab test {lab_code} has {num_units} units!")
    else:
        raise NotImplementedError(f"Data source '{data_source}' not implemented.")

def plot_histograms_by_unit(cbc_df: pd.DataFrame) -> None:
    """
    Plot histograms for numeric values grouped by units for each unique lab code.
    Only plots when there is more than one unique unit.
    """
    for code in cbc_df['code'].unique():
        cbc_sub = cbc_df[cbc_df['code'] == code]
        unique_units = cbc_sub['unit'].dropna().unique()

        # Skip plotting if only one unit exists
        if len(unique_units) <= 1:
            continue

        # Create a subplot for each unit
        fig, axes = plt.subplots(1, len(unique_units), figsize=(5 * len(unique_units), 2))
        axes = axes if len(unique_units) > 1 else [axes]

        for idx, unit in enumerate(unique_units):
            sns.histplot(cbc_sub[cbc_sub['unit'] == unit], x='numeric_value', kde=True, bins=30, ax=axes[idx])
            axes[idx].set_title(f"Code: {code} | Unit: {unit}")
            axes[idx].set_xlabel("Numeric Value")
            axes[idx].set_ylabel("Frequency")
        
        plt.tight_layout()
        plt.show()

def filter_cbc_tests(df):
    """
    For each subject, keep only those who have at least 5 records of the specified blood test.
    Additionally, each measurement must be at least 90 days apart.
    """
    df['time'] = pd.to_datetime(df['time'])

    print("Taking mean of duplicate lab tests per time...")
    agg_df = df.groupby(["subject_id", "code", "time", "unit"])['numeric_value'].mean().reset_index()
    print(f"Before dropping duplicates, had: {len(df)} records")
    print(f"Dropped duplicates, now have: {len(agg_df)} records")

    print("Removing patients with less than 2 tests...")
    filter_df = agg_df.groupby(["subject_id", "code"]).filter(lambda x: len(x) >= 2)
    print(f"After filtering, have: {len(filter_df)} records")
    return filter_df

def cbc_summary(df):
    """
    Returns a summary DataFrame with:
        - subject_id
        - code
        - num_tests_taken
        - time between first and last test
        - average time between tests
        - min time between tests
        - max time between tests
    """

    def compute_summary(group):
        time_diffs = group['time'].diff().dropna().dt.total_seconds() / (60 * 60 * 24)  # Convert to days
        return pd.Series({
            'num_tests_taken': len(group),
            'time_between_first_and_last': (group['time'].max() - group['time'].min()).days,
            'avg_time_between_tests': f"{time_diffs.mean():.2f}" if not time_diffs.empty else None,
            'min_time_between_tests': f"{time_diffs.min():.2f}" if not time_diffs.empty else None,
            'max_time_between_tests': f"{time_diffs.max():.2f}" if not time_diffs.empty else None
        })

    summary_df = df.groupby(['subject_id', 'code']).apply(compute_summary).reset_index()
    return summary_df

def find_death():
    """
    Find the date of death for each subject.
    """
    pass

def find_diagnosis():
    """
    Find the diagnosis for each subject.
    """
    pass


In [197]:
pwd

UsageError: CWD no longer exists - please use %cd to change directory.


In [184]:
DATA_PATH = "/Users/aashnashah/Desktop/ssh_mount/data/EHRSHOT/meds_omop_ehrshot/"
df = load_dataset(DATA_PATH)
print(df.head())
print(df.shape)

# Extract demographics and filter valid subjects
demo_df = filter_demographics(df)
valid_subjects = set(demo_df['subject_id'])
        
# Remove subjects without valid demographics
filtered_df = df[df['subject_id'].isin(valid_subjects)]
bmi_df = standardize_and_compute_bmi(filtered_df)
valid_subjects = set(bmi_df['subject_id'])
filtered_df = df[df['subject_id'].isin(valid_subjects)]

death_df = df[df['table'] == 'death']

cbc_df = extract_cbc_tests(filtered_df)
check_units(cbc_df)
plot_histograms_by_unit(cbc_df)
        
label = cbc_df.code.unique()[0]
filtered_df = filter_cbc_tests(cbc_df)
cbc_summary = cbc_filter_summary(filtered_df)

print(f"Final CBC dataset shape: {cbc_df.shape}")

FileNotFoundError: [Errno 2] No such file or directory

In [49]:
cbc_df.head()

,subject_id,time,code,numeric_value,care_site_id,clarity_table,end,note_id,provider_id,table,text_value,unit,visit_id
239,115967562,2021-09-01 10:35:00,LOINC/4544-3,46.400002,None,shc_order_results,NaT,None,6773675,measurement,None,%,185409752
250,115967562,2021-09-01 10:35:00,LOINC/718-7,14.900000,None,shc_order_results,NaT,None,6773675,measurement,None,g/dL,185409752
256,115967562,2021-09-01 10:35:00,LOINC/777-3,225.000000,None,shc_order_results,NaT,None,6773675,measurement,None,K/uL,185409752
257,115967562,2021-09-01 10:35:00,LOINC/785-6,32.700001,None,shc_order_results,NaT,None,6773675,measurement,None,pg,185409752
258,115967562,2021-09-01 10:35:00,LOINC/786-4,32.099998,None,shc_order_results,NaT,None,6773675,measurement,None,g/dL,185409752


In [ ]:
def print_labdf_stats(df, label_col):
        """
        Print basic statistics about the dataframe (useful to understand
        the data after each processing step
        """
        print(df.head())
        print(df.shape)
        print("Number of entries:", len(df))
        print("Number of patients:", len(df.subject_id.unique()))
        print("Number of lab items labels:", len(df[label_col].unique()))
        print("Extract most common lab codes...")
    

# Take mean of duplicate lab tests per time
print("Take mean of duplicate lab tests per time...")
print(filter_df[["subject_id", labid_col, "charttime", labunit_col, label_col]].drop_duplicates())
print(filter_df.groupby(["subject_id", labid_col, "charttime", labunit_col, label_col]).mean())
filter_df = filter_df.groupby(["subject_id", labid_col, "charttime", labunit_col, label_col]).mean().reset_index()

print("Keep patients with more than one visit...")
pt_count_visits = filter_df[["subject_id", "charttime"]].drop_duplicates().groupby("subject_id").count()
print(pt_count_visits)

print(pt_count_visits.sort_values(by = "charttime", ascending = True))
pt_count_visits = pt_count_visits[pt_count_visits["charttime"] > 1]
print(pt_count_visits.sort_values(by = "charttime", ascending = True))
filter_df = filter_df[filter_df.subject_id.isin(pt_count_visits.index.tolist())]
print_labdf_stats(filter_df, label_col)

In [16]:
data_dir = '../../CLEF/data_prep/data/mimic/'
mimic_lab = pd.read_csv(os.path.join(data_dir, 'filtered_lab_code_data.csv'), sep = '\t', nrows = 100)
mimic_lab.action_cat.unique()
mimic_lab.head()

,subject_id,itemid,charttime,valueuom,label,valuenum,action,action_cat
0,10000032,50868,2180-03-23 11:51:00,mEq/L,Anion Gap,12.0,NaN,NaN
1,10000032,50882,2180-03-23 11:51:00,mEq/L,Bicarbonate,27.0,NaN,NaN
2,10000032,50902,2180-03-23 11:51:00,mEq/L,Chloride,101.0,NaN,NaN
3,10000032,50912,2180-03-23 11:51:00,mg/dL,Creatinine,0.4,NaN,NaN
4,10000032,50931,2180-03-23 11:51:00,mg/dL,Glucose,95.0,NaN,NaN


In [29]:
ehr_asset = pd.read_csv('../../data/EHRSHOT/EHRSHOT_ASSETS/data/ehrshot.csv', nrows=100000)